# Mid-Term Practical Evaluation — Machine Learning
**Universidad Internacional del Ecuador — Escuela de Ciencias de la Computación**
**Autor:** Daniel Sozoranga · **Docente:** Ing. Jonathan E. Tito O., MSc.
**Duration:** 90 min · **Individual** · **Total:** 12 pts

---

---
# PART A — Setup & Diagnostics (2.5 pts, ~20 min)

## Task 1 — Environment check (0.5 pts)
Run the cell below. If you see version numbers printed, you are ready.

In [ ]:
import sys
import pandas as pd
import sklearn
import numpy as np

print(f"Python ......... {sys.version.split()[0]}")
print(f"pandas ......... {pd.__version__}")
print(f"scikit-learn ... {sklearn.__version__}")
print(f"numpy .......... {np.__version__}")
print("\nEnvironment OK — you can proceed.")

## Task 2 — Fix the bug in `src/scale.py` (1 pt)

**Bug encontrado:** el numerador decía `arr - col_max` en lugar de `arr - col_min`.
- Con `col_max`: cuando `x = col_min` → `(col_min - col_max)/(col_max - col_min) = -1` ❌
- Con `col_min`: cuando `x = col_min` → `0 / rango = 0` ✅

**Fix aplicado en `src/scale.py` (línea del return):**
```python
# BUG: el numerador usaba col_max en lugar de col_min, produciendo rango [-1, 0].
# FIX: reemplazar col_max por col_min para obtener rango [0, 1].
return (arr - col_min) / (col_max - col_min)
```

In [ ]:
# Verification cell — do not modify
import importlib
from src import scale
importlib.reload(scale)
import numpy as np

test_data = np.array([10, 20, 30, 40, 50])
scaled = scale.min_max_scale(test_data)

assert scaled.min() == 0.0, f"min should be 0, got {scaled.min()}"
assert scaled.max() == 1.0, f"max should be 1, got {scaled.max()}"
print(f"Scaled output: {scaled}")
print("Bug fix verified: OK")

## Task 3 — Pandas indexing (1 pt)

In [ ]:
import pandas as pd

df_small = pd.DataFrame({
    "alcohol":   [9.5, 10.2, 11.0, 12.8, 9.9],
    "ph":        [3.4, 3.2, 3.5, 3.1, 3.3],
    "category":  pd.Categorical(["train", "train", "test", "test", "train"]),
})
df_small

In [ ]:
# Task 3a: dtype de la columna 'category'
print(df_small["category"].dtype)
# Es CategoricalDtype (NO 'object') porque se creó con pd.Categorical().
# Pandas almacena los valores como códigos enteros + índice de categorías,
# lo que reduce memoria y permite ordenamiento semántico por categoría.

# Task 3b: label-based indexing con .loc — usa el LABEL del índice, no la posición
# El índice por defecto es 0,1,2,3,4 → label 3 = cuarta fila → alcohol = 12.8
target_value = df_small.loc[3, "alcohol"]
print(f"target_value = {target_value}")

---
# PART B — Linear Regression (5 pts, ~35 min)

## Load the dataset
Run the cell below. **Do not modify it.**

In [ ]:
import pandas as pd
df = pd.read_csv("data/wine_quality.csv" if __import__("os").path.exists("data/wine_quality.csv") else "../data/wine_quality.csv")
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

## Task 4 — Quick EDA (1 pt)
Dos plots: distribución del target `quality` + heatmap de correlación.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: distribución del target
quality_counts = df["quality"].value_counts().sort_index()
axes[0].bar(quality_counts.index.astype(str), quality_counts.values,
            color="steelblue", edgecolor="white")
axes[0].set_xlabel("Calidad (quality)")
axes[0].set_ylabel("Número de muestras")
axes[0].set_title("Distribución del target: quality")
for i, (q, v) in enumerate(quality_counts.items()):
    axes[0].text(i, v + 3, str(v), ha="center", fontsize=9, fontweight="bold")

# Plot 2: heatmap de correlación
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True,
            ax=axes[1], cbar_kws={"shrink": 0.8})
axes[1].set_title("Mapa de correlación (Pearson)")

plt.tight_layout()
plt.show()

**Observaciones:**

- **(a)** La feature con mayor correlación con `quality` es **`alcohol`** (r ≈ 0.36): a mayor contenido alcohólico, mayor puntuación de calidad.
- **(b)** El dataset presenta **desbalance de clases severo**: las calidades extremas (3 → 10 muestras, 8 → 18 muestras) tienen muy pocas instancias frente a las centrales (5 → 681, 6 → 638), lo que puede sesgar el modelo a predecir valores medios.

## Task 5 — Preprocessing Pipeline (2 pts)

**Justificación del scaler:** uso `StandardScaler` porque las features tienen escalas muy distintas (`total_sulfur_dioxide` ~ 46 vs `citric_acid` ~ 0.27). Estandarizar a media=0, std=1 hace los coeficientes comparables y estabiliza el descenso de gradiente. `MinMaxScaler` sería sensible a los outliers presentes en el dataset.

**El Pipeline se fittea SOLO en `X_train`** — el scaler aprende μ y σ únicamente de train y los aplica a test vía `.transform()`, evitando data leakage.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# 1. X e y
X = df.drop(columns=["quality"])
y = df["quality"]

# 2. Split 80/20, random_state=42 fijo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 3. Pipeline: StandardScaler (justificado arriba) + LinearRegression
pipe_linear = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model",  LinearRegression()),
])

# 4. Fit SOLO sobre training set — el scaler NO ve X_test
pipe_linear.fit(X_train, y_train)

print(f"Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas")
print(f"Pasos del pipeline: {[name for name, _ in pipe_linear.steps]}")

## Task 6 — Train & evaluate linear regression (2 pts)

In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

y_pred = pipe_linear.predict(X_test)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 45)
print("LINEAR REGRESSION — Test set metrics")
print("=" * 45)
print(f"  R²   : {r2:.4f}")
print(f"  MAE  : {mae:.4f}  (puntos de calidad)")
print(f"  RMSE : {rmse:.4f}")

**Interpretación:**

El modelo explica solo el **31.6% de la varianza** del target (R² ≈ 0.32), con un error promedio de **±0.54 puntos** (MAE); en una escala de 3 a 8, esto indica que la relación fisicoquímica→calidad existe pero es débil, haciendo al modelo poco usable para predicciones precisas — el error típico equivale a confundir un vino de calidad 5 con uno de 6.

---
# PART C — Logistic Regression (4 pts, ~25 min)

## Task 7 — Engineer a binary target (0.5 pts)
Crear `is_good` = 1 si `quality >= 7`, else 0.

In [ ]:
df["is_good"] = (df["quality"] >= 7).astype(int)

counts = df["is_good"].value_counts().sort_index()
print("Distribución de is_good:")
print(f"  Clase 0 (no-good, quality < 7) : {counts[0]:>4} muestras ({counts[0]/len(df)*100:.1f}%)")
print(f"  Clase 1 (good,   quality >= 7) : {counts[1]:>4} muestras ({counts[1]/len(df)*100:.1f}%)")
# → Clase 0: 1382 (86.4%), Clase 1: 217 (13.6%)
# Desbalance severo: los vinos buenos son solo el 13.6% del dataset.
# Esto afectará directamente F1 y la confusion matrix (ver Task 9).

## Task 8 — Train logistic regression with L2 (2 pts)

**L2 es el `penalty` por defecto** en sklearn `LogisticRegression` (`penalty="l2"`): penaliza `λ‖w‖²` en la log-likelihood, comprimiendo los coeficientes hacia 0. Es especialmente útil aquí porque `free_sulfur_dioxide` y `total_sulfur_dioxide` están correlacionadas y L2 reparte el peso entre ambas en lugar de asignarlo todo a una.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

X2 = df.drop(columns=["quality", "is_good"])
y2 = df["is_good"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42
)

# penalty="l2" es el default — lo explicito para dejar claro que L2 es la regularización
pipe_logistic = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(penalty="l2", max_iter=1000, random_state=42)),
])

pipe_logistic.fit(X2_train, y2_train)

y2_pred  = pipe_logistic.predict(X2_test)
y2_proba = pipe_logistic.predict_proba(X2_test)[:, 1]

acc = accuracy_score(y2_test, y2_pred)
f1  = f1_score(y2_test, y2_pred)
auc = roc_auc_score(y2_test, y2_proba)
cm  = confusion_matrix(y2_test, y2_pred)

print("=" * 45)
print("LOGISTIC REGRESSION (L2) — Test set metrics")
print("=" * 45)
print(f"  Accuracy : {acc:.4f}")
print(f"  F1 Score : {f1:.4f}")
print(f"  ROC-AUC  : {auc:.4f}")
print()
print("Confusion Matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["no-good (0)", "good (1)"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix\nAcc={acc:.3f} | F1={f1:.3f} | AUC={auc:.3f}")
ax.grid(False)
plt.tight_layout()
plt.show()

## Task 9 — Interpret class imbalance (1 pt)

**Respuesta:**

- **(a)** La clase **1 (good wines)** es más difícil de predecir: el modelo detecta solo 7 de los 45 vinos buenos en el test set, mientras clasifica correctamente 269 de los 275 no-buenos.
- **(b)** La confusion matrix muestra **38 falsos negativos** (buenos vinos clasificados como no-buenos) frente a solo **7 verdaderos positivos**: el desbalance 86/14 hace que el modelo aprenda a predecir casi siempre clase 0, logrando alta Accuracy (86%) pero F1 de apenas 0.24 — la Accuracy engaña; F1 y la matriz revelan que el modelo no "ve" la clase minoritaria.

---
# PART D — Reflection & Submit (1 pt, ~10 min)

## Task 10 — Written reflection (0.5 pts)

**Reflexión:**

Con una hora más, lo primero que cambiaría es aplicar `class_weight="balanced"` en `LogisticRegression` (concepto de clase: **desbalance de clases**): el ratio 86/14 hace que el modelo ignore la clase minoritaria, como evidencia la confusion matrix con 38 falsos negativos frente a solo 7 verdaderos positivos. Con `class_weight="balanced"`, sklearn ajusta los pesos de cada clase inversamente proporcional a su frecuencia, forzando al modelo a penalizar más los errores sobre vinos buenos.

Después exploraría el hiperparámetro `C` de la regresión logística con validación cruzada (concepto de clase: **regularización y overfitting**): el `C=1.0` por defecto puede estar sobreregularizando features con señal real como `alcohol` (r≈0.36 con quality). Un sweep `C ∈ {0.01, 0.1, 1, 10, 100}` con `GridSearchCV` mostraría la curva underfitting→overfitting y permitiría escoger el punto óptimo.

Finalmente, para la regresión lineal trataría los **outliers** del target (concepto de clase: **outliers distorsionan la recta de mínimos cuadrados**): quality=3 tiene solo 10 muestras y quality=8 solo 18, y esos puntos extremos arrastran la recta de regresión en una dirección que no representa el comportamiento general del dataset.

## Task 11 — Commit, push, submit (0.5 pts)

```bash
git add .
git commit -m "Final exam submission"
git push
```

Pegar la URL del repositorio en Canvas antes de que expire el timer de 90 minutos.
**El timestamp del commit de Git es la hora oficial de entrega.**